In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/mubhinbasnet/Desktop/student-dropout-prediction


In [7]:
import mlflow
from pathlib import Path

# Use SQLite database for MLflow tracking
tracking_db = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{tracking_db.as_posix()}"
)

# Create or select experiment
mlflow.set_experiment("Student Dropout Prediction")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment configured successfully!")

MLflow tracking URI: sqlite:////Users/mubhinbasnet/Desktop/student-dropout-prediction/mlflow.db
Experiment configured successfully!


In [9]:
import mlflow

# Make sure we are using the correct SQLite database
mlflow.set_tracking_uri(
    f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"
)

# Select the correct experiment
experiment = mlflow.set_experiment("Student Dropout Prediction")

experiments = [
    {
        "name": "Dummy Classifier",
        "params": {"model_type": "DummyClassifier", "strategy": "most_frequent"},
        "metrics": {
            "accuracy": 0.6093, "precision": 0.0,
            "recall": 0.0, "f1": 0.0,
            "roc_auc": 0.5000, "pr_auc": 0.3907
        }
    },
    {
        "name": "Business Rule",
        "params": {"model_type": "Business Rule"},
        "metrics": {
            "accuracy": 0.7332, "precision": 0.8673,
            "recall": 0.3744, "f1": 0.5231,
            "roc_auc": 0.6689, "pr_auc": 0.5692
        }
    },
    {
        "name": "Logistic Regression",
        "params": {"model_type": "LogisticRegression", "C": 1.0},
        "metrics": {
            "accuracy": 0.8107, "precision": 0.8197,
            "recall": 0.6608, "f1": 0.7317,
            "roc_auc": 0.8529, "pr_auc": 0.8318
        }
    },
    {
        "name": "Random Forest",
        "params": {"model_type": "RandomForestClassifier", "n_estimators": 100},
        "metrics": {
            "accuracy": 0.7780, "precision": 0.8182,
            "recall": 0.5551, "f1": 0.6614,
            "roc_auc": 0.8376, "pr_auc": 0.8148
        }
    },
    {
        "name": "Tuned Logistic Regression",
        "params": {"model_type": "LogisticRegression", "C": 0.1, "class_weight": "balanced"},
        "metrics": {
            "accuracy": 0.7849, "precision": 0.7277,
            "recall": 0.7181, "f1": 0.7228,
            "roc_auc": 0.8503, "pr_auc": 0.8263
        }
    },
    {
        "name": "Tuned Random Forest",
        "params": {
            "model_type": "RandomForestClassifier",
            "n_estimators": 200,
            "max_depth": "None",
            "min_samples_leaf": 2,
            "class_weight": "balanced"
        },
        "metrics": {
            "accuracy": 0.7762, "precision": 0.7215,
            "recall": 0.6960, "f1": 0.7085,
            "roc_auc": 0.8388, "pr_auc": 0.8168
        }
    }
]

for item in experiments:
    with mlflow.start_run(
        experiment_id=experiment.experiment_id,
        run_name=item["name"]
    ):
        mlflow.log_params(item["params"])
        mlflow.log_metrics(item["metrics"])
        mlflow.set_tag("evaluation_stage", "validation")
        mlflow.set_tag("result_type", "previously_recorded_result")

        print("Successfully logged:", item["name"])

print("\nSix experiments logged!")

Successfully logged: Dummy Classifier
Successfully logged: Business Rule
Successfully logged: Logistic Regression
Successfully logged: Random Forest
Successfully logged: Tuned Logistic Regression
Successfully logged: Tuned Random Forest

Six experiments logged!


In [10]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print("Total runs:", len(runs))

display(
    runs[[
        "tags.mlflow.runName",
        "metrics.accuracy",
        "metrics.f1",
        "metrics.roc_auc"
    ]]
)

Total runs: 6


,tags.mlflow.runName,metrics.accuracy,metrics.f1,metrics.roc_auc
0,Tuned Random Forest,0.7762,0.7085,0.8388
1,Tuned Logistic Regression,0.7849,0.7228,0.8503
2,Random Forest,0.7780,0.6614,0.8376
3,Logistic Regression,0.8107,0.7317,0.8529
4,Business Rule,0.7332,0.5231,0.6689
5,Dummy Classifier,0.6093,0.0000,0.5000


In [11]:
import mlflow
from pathlib import Path

# Record final test evaluation
with mlflow.start_run(
    experiment_id=experiment.experiment_id,
    run_name="Final Model - Test Evaluation"
):

    mlflow.log_params({
        "model_type": "Logistic Regression",
        "C": 1.0,
        "class_weight": "None",
        "max_iter": 2000,
        "random_state": 42,
        "test_records": 726,
        "decision_threshold": 0.50
    })

    mlflow.log_metrics({
        "accuracy": 0.7837,
        "precision": 0.7510,
        "recall": 0.6690,
        "f1": 0.7076,
        "roc_auc": 0.8535,
        "pr_auc": 0.8201
    })

    mlflow.set_tag("evaluation_stage", "final_test")
    mlflow.set_tag("selected_model", "true")

    # Attach saved model
    mlflow.log_artifact(
        str(PROJECT_ROOT / "models" / "student_dropout_model.joblib")
    )

    # Attach model metadata
    mlflow.log_artifact(
        str(PROJECT_ROOT / "models" / "model_metadata.json")
    )

print("Final test evaluation recorded successfully!")

Final test evaluation recorded successfully!


In [12]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"]
)

print("Total experiment runs:", len(runs))

display(
    runs[[
        "tags.mlflow.runName",
        "tags.evaluation_stage",
        "metrics.accuracy",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
        "metrics.pr_auc"
    ]]
)

Total experiment runs: 7


,tags.mlflow.runName,tags.evaluation_stage,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc,metrics.pr_auc
0,Logistic Regression,validation,0.8107,0.8197,0.6608,0.7317,0.8529,0.8318
1,Tuned Logistic Regression,validation,0.7849,0.7277,0.7181,0.7228,0.8503,0.8263
2,Tuned Random Forest,validation,0.7762,0.7215,0.6960,0.7085,0.8388,0.8168
3,Final Model - Test Evaluation,final_test,0.7837,0.7510,0.6690,0.7076,0.8535,0.8201
4,Random Forest,validation,0.7780,0.8182,0.5551,0.6614,0.8376,0.8148
5,Business Rule,validation,0.7332,0.8673,0.3744,0.5231,0.6689,0.5692
6,Dummy Classifier,validation,0.6093,0.0000,0.0000,0.0000,0.5000,0.3907
